# Guided Project: PDF-Based Knowledge Base RAG System
## Solution Notebook  -  trainer / reference use only

Complete reference implementation for every task. Keep this out of the participant solution-sharing area.

In [ ]:
# Task 0 - Environment preflight (run this first)
# The VM is preconfigured. This cell only VERIFIES that everything is in place.
import importlib, os, sys

REQUIRED = [
    "langchain", "langchain_openai", "langchain_community",
    "langchain_text_splitters", "langchain_chroma", "chromadb",
    "pypdf", "dotenv",
]
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]

from dotenv import load_dotenv
load_dotenv("../05_CONFIG/.env")   # vendor-managed key; safe no-op if absent

key = os.getenv("OPENAI_API_KEY", "")
data_ok = os.path.isfile("../04_DATA/sample_knowledge_base.pdf")

print("packages   :", "OK" if not missing else f"MISSING -> {missing}")
print("OPENAI key  :", "OK" if key.startswith("sk-") else "NOT CONFIGURED - contact the lab administrator")
print("sample PDF  :", "OK" if data_ok else "MISSING at ../04_DATA/sample_knowledge_base.pdf")

assert not missing, f"Preconfigured VM is missing packages: {missing}"
assert data_ok, "Supplied PDF not found."
print("\nPreflight passed - continue with Task 1.")

## Task 1 - PDF Loading

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "../04_DATA/sample_knowledge_base.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Number of pages:", len(documents))
print("\n--- First page ---")
print(documents[0].page_content[:800])
print("\nMetadata:", documents[0].metadata)

## Task 2 - Document Chunking (1000 / 200)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)
chunks = text_splitter.split_documents(documents)

print("Pages :", len(documents))
print("Chunks:", len(chunks))
for i, ch in enumerate(chunks[:2]):
    print(f"\n--- Chunk {i + 1} ---")
    print(ch.page_content[:500])
    print("Metadata:", ch.metadata)

## Task 3 - Embeddings + persistent ChromaDB

In [ ]:
import os
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

PERSIST_DIR = "../chroma_db"
COLLECTION = "pdf_kb"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Load the store if it already exists, otherwise build it once.
if os.path.isdir(PERSIST_DIR) and os.listdir(PERSIST_DIR):
    vectorstore = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embeddings,
        collection_name=COLLECTION,
    )
    print("Loaded existing Chroma store.")
else:
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=PERSIST_DIR,
        collection_name=COLLECTION,
    )
    print("Built Chroma store and persisted to", PERSIST_DIR)

In [ ]:
query = "How many casual leave days are provided each year?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Retrieved chunk {i + 1} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:400])

## Task 4 - Grounded RAG QA with GPT-4o-mini

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template(
    """You are a PDF question-answering assistant.

Answer the question using ONLY the information in the context below.
Do not use outside knowledge.
If the answer cannot be found in the context, reply exactly:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:"""
)

In [ ]:
def answer_question(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    messages = prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    pages = sorted({d.metadata.get("page", "?") for d in docs})
    return response.content, pages

for q in ["What is the standard notice period for regular full-time employees?",
          "What class of air travel is standard for domestic business trips?",
          "Who is the current Prime Minister of India?"]:
    ans, pages = answer_question(q)
    print("Q:", q)
    print("A:", ans)
    print("Pages:", pages)
    print("-" * 70)

## Task 5 - Interactive application

In [ ]:
# The full interactive app lives in ../03_SOLUTION_GUIDE/rag_app.py
# (participant version to complete: ../02_STARTER_CODE/rag_app.py).
# Run it from a terminal:  cd ../03_SOLUTION_GUIDE && python rag_app.py
print(open('../03_SOLUTION_GUIDE/rag_app.py').read())

## Validation

Run the automated harness against `../04_DATA/RAG_Question_Set.pdf`:

```bash
python ../scripts/validate_solution.py
```

It checks that grounded questions get non-refusal answers with source pages, and that every negative / out-of-context question triggers the exact refusal sentence.